In [ ]:
# Cell 1 — Imports & Config
import os, gc, glob, warnings
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import psutil
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# --- Paths ---
DATA_NEW_DIR    = "data_new"                # 47 monthly BTS folders (2022-2025)
AIRPORTS_PATH   = "Data/airports.csv"       # iata, lat, lon
WEATHER_CACHE   = "Data/weather_cache"      # cached hourly weather per airport
MODELS_DIR      = "models"
PROCESSED_OUT   = "Data/processed_flights_sample.parquet"

CLASSIFIER_PATH   = f"{MODELS_DIR}/rf_classifier_sample.joblib"
REGRESSOR_PATH    = f"{MODELS_DIR}/rf_regressor_sample.joblib"
ENCODER_PATH      = f"{MODELS_DIR}/ordinal_encoder.joblib"
TOP_ORIG_PATH     = f"{MODELS_DIR}/top_orig.joblib"
TOP_DEST_PATH     = f"{MODELS_DIR}/top_dest.joblib"
ROUTE_AVG_PATH    = f"{MODELS_DIR}/route_avg_delay.joblib"
PREPROCESSOR_PATH = f"{MODELS_DIR}/preprocessor_sample.joblib"

# --- Settings ---
ROWS_PER_FILE = 21000   # 21K x 47 files = ~1M rows total
TOP_K         = 50      # top airports to keep individually; rest become 'OTHER'
RANDOM_STATE  = 42

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(WEATHER_CACHE, exist_ok=True)

print('LightGBM:', lgb.__version__)
print(f'RAM available: {psutil.virtual_memory().available/1e9:.1f} GB')

In [ ]:
# Cell 2 — Load all 47 monthly BTS files (sampled ~21K rows each = ~1M total)

csv_files = sorted(glob.glob(f"{DATA_NEW_DIR}/**/*.csv", recursive=True))
print(f"Found {len(csv_files)} CSV files")

USE_COLS = ['Year','Month','DayofMonth','DayOfWeek','FlightDate',
            'Reporting_Airline','Origin','Dest',
            'CRSDepTime','CRSArrTime','ArrDelay',
            'Distance','Cancelled','Diverted']

chunks = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, low_memory=False, usecols=USE_COLS, nrows=ROWS_PER_FILE)
        chunks.append(tmp)
    except Exception as e:
        print(f"  Skipping {f}: {e}")

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols")
print(f"Years: {sorted(df['Year'].unique())}")
print(f"RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB")

In [ ]:
# Cell 3 — Clean
df = df[(df['Cancelled'] != 1) & (df['Diverted'] != 1)].copy()
df['ArrDelay'] = pd.to_numeric(df['ArrDelay'], errors='coerce')
df = df[df['ArrDelay'].notna()].copy()
df['Origin']  = df['Origin'].astype(str).str.strip().str.upper()
df['Dest']    = df['Dest'].astype(str).str.strip().str.upper()
df['carrier'] = df['Reporting_Airline'].astype(str).str.strip().str.upper()
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')

print(f"After cleaning: {len(df):,} rows")
print("Carriers:", sorted(df['carrier'].unique()))

In [ ]:
# Cell 4 — Feature engineering
# These features are now ALL properly available in the new data format!

def hhmm_to_hour(x):
    try: return int(float(x)) // 100
    except: return np.nan

df['dep_hour']  = df['CRSDepTime'].apply(hhmm_to_hour)   # scheduled dep hour (0-23)
df['arr_hour']  = df['CRSArrTime'].apply(hhmm_to_hour)   # scheduled arr hour (0-23)
df['month']     = df['Month'].astype(float)               # 1-12 (seasonality)
df['dayofweek'] = df['DayOfWeek'].astype(float)           # 1=Mon, 7=Sun
df['distance']  = pd.to_numeric(df['Distance'], errors='coerce')

# Top-K airports
top_orig_list = df['Origin'].value_counts().nlargest(TOP_K).index.tolist()
top_dest_list = df['Dest'].value_counts().nlargest(TOP_K).index.tolist()
df['origin_topK'] = df['Origin'].where(df['Origin'].isin(top_orig_list), other='OTHER')
df['dest_topK']   = df['Dest'].where(df['Dest'].isin(top_dest_list),   other='OTHER')

# Targets
df['DELAY_MINUTES'] = df['ArrDelay']
def delay_class(x):
    if x <= 15: return 0
    if x <= 59: return 1
    return 2
df['DELAY_CLASS'] = df['DELAY_MINUTES'].apply(delay_class).astype(int)

print("Features engineered — non-null counts:")
for col in ['dep_hour','arr_hour','month','dayofweek','distance']:
    print(f"  {col}: {df[col].notna().sum():,} / {len(df):,}")
print("\nDelay class distribution:")
print(df['DELAY_CLASS'].value_counts().sort_index())

In [ ]:
# Cell 5 — Fetch weather from Meteostat (cached per airport)
# Weather is fetched once per airport and cached as parquet.
# Re-running this cell is fast — it just loads from cache.
from meteostat import Stations, Hourly

airports_df = pd.read_csv(AIRPORTS_PATH)
airports_df.columns = [c.lower() for c in airports_df.columns]
airports_df['iata'] = airports_df['iata'].astype(str).str.strip().str.upper()

all_airports = list(set(top_orig_list + top_dest_list) - {'OTHER'})
airport_coords = airports_df[airports_df['iata'].isin(all_airports)].set_index('iata')[['lat','lon']]
print(f"Fetching weather for {len(airport_coords)} airports (2022-2025)...")

START = datetime(2022, 1, 1)
END   = datetime(2025, 11, 30)

weather_frames = []
missing_airports = []

for iata, row in airport_coords.iterrows():
    cache_file = f"{WEATHER_CACHE}/{iata}.parquet"
    if os.path.exists(cache_file):
        weather_frames.append(pd.read_parquet(cache_file))
        continue
    try:
        stations = Stations().nearby(row['lat'], row['lon']).fetch(1)
        if stations.empty:
            missing_airports.append(iata); continue
        wx = Hourly(stations.index[0], START, END).fetch()
        if wx.empty:
            missing_airports.append(iata); continue
        wx = wx[['temp','wspd','prcp','snow','coco']].reset_index()
        wx.columns = ['time','temp','wspd','prcp','snow','coco']
        wx['iata'] = iata
        wx['date'] = wx['time'].dt.date.astype(str)
        wx['hour'] = wx['time'].dt.hour
        wx = wx.drop(columns=['time'])
        wx.to_parquet(cache_file, index=False)
        weather_frames.append(wx)
    except Exception as e:
        missing_airports.append(iata)

weather_df = pd.concat(weather_frames, ignore_index=True)
print(f"Weather loaded for {len(weather_frames)} airports, {len(weather_df):,} hourly records")
if missing_airports:
    print(f"Missing weather for: {missing_airports}")

In [ ]:
# Cell 6 — Merge weather onto flights
# Match: origin airport + flight date + departure hour

df['date'] = df['FlightDate'].dt.date.astype(str)

df = df.merge(
    weather_df.rename(columns={'iata': 'Origin'}),
    left_on  = ['Origin', 'date', 'dep_hour'],
    right_on = ['Origin', 'date', 'hour'],
    how='left'
).drop(columns=['hour'], errors='ignore')

# Fill missing weather with training medians
for col in ['temp','wspd','prcp','snow','coco']:
    df[col] = df[col].fillna(df[col].median())

print(f"Weather match rate: {df['wspd'].notna().mean():.1%}")
print(f"Shape after merge: {df.shape}")
print(df[['Origin','date','dep_hour','temp','wspd','prcp','snow','coco']].head(3))

In [ ]:
# Cell 7 — Train/test split

FEATURE_COLS = ['dep_hour','arr_hour','month','dayofweek','distance',
                'temp','wspd','prcp','snow','coco',
                'origin_topK','dest_topK','carrier']

X = df[FEATURE_COLS].copy()
y_cls = df['DELAY_CLASS']
y_reg = df['DELAY_MINUTES']

X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE
)
y_train_reg = y_reg.loc[y_train_cls.index]
y_test_reg  = y_reg.loc[y_test_cls.index]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

In [ ]:
# Cell 8 — Route average delay feature (train-only to prevent leakage)

train_tmp = X_train[['origin_topK','dest_topK']].copy()
train_tmp['DELAY_MINUTES'] = y_train_reg.values

route_avg = (
    train_tmp.groupby(['origin_topK','dest_topK'])['DELAY_MINUTES']
    .mean().reset_index()
    .rename(columns={'DELAY_MINUTES': 'route_avg_delay'})
)
global_avg = float(train_tmp['DELAY_MINUTES'].mean())

def add_route_avg(X_df):
    out = X_df.merge(route_avg, on=['origin_topK','dest_topK'], how='left')
    out['route_avg_delay'] = out['route_avg_delay'].fillna(global_avg)
    return out

X_train = add_route_avg(X_train)
X_test  = add_route_avg(X_test)

FEATURE_COLS_FULL = FEATURE_COLS + ['route_avg_delay']
print("Top 10 most delayed routes:")
print(route_avg.nlargest(10, 'route_avg_delay').to_string(index=False))

In [ ]:
# Cell 9 — Encode categoricals + fill numeric NaNs

CAT_COLS = ['origin_topK','dest_topK','carrier']
NUM_COLS = [c for c in FEATURE_COLS_FULL if c not in CAT_COLS]

num_medians = {}
for col in NUM_COLS:
    med = float(X_train[col].median())
    num_medians[col] = med
    X_train[col] = X_train[col].fillna(med)
    X_test[col]  = X_test[col].fillna(med)

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[CAT_COLS] = enc.fit_transform(X_train[CAT_COLS]).astype(int)
X_test[CAT_COLS]  = enc.transform(X_test[CAT_COLS]).astype(int)

for col in CAT_COLS:
    X_train[col] = pd.Categorical(X_train[col])
    X_test[col]  = pd.Categorical(X_test[col])

X_train = X_train[FEATURE_COLS_FULL]
X_test  = X_test[FEATURE_COLS_FULL]

print(f"Final features ({len(FEATURE_COLS_FULL)}): {FEATURE_COLS_FULL}")
print(f"X_train shape: {X_train.shape}")
print(f"RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB")

In [ ]:
# Cell 10 — Train LightGBM Classifier
from lightgbm import LGBMClassifier

clf = LGBMClassifier(
    n_estimators      = 500,
    num_leaves        = 63,
    learning_rate     = 0.05,
    class_weight      = 'balanced',
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1
)
clf.fit(
    X_train, y_train_cls,
    eval_set=[(X_test, y_test_cls)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred_cls = clf.predict(X_test)
print("\n=== Classification Report ===")
print(classification_report(y_test_cls, y_pred_cls,
      target_names=['On-time','Minor delay','Major delay']))

In [ ]:
# Cell 11 — Train LightGBM Regressor
from lightgbm import LGBMRegressor

reg = LGBMRegressor(
    n_estimators      = 500,
    num_leaves        = 63,
    learning_rate     = 0.05,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose            = -1
)
reg.fit(
    X_train, y_train_reg,
    eval_set=[(X_test, y_test_reg)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred_reg = reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2   = r2_score(y_test_reg, y_pred_reg)
print(f"\n=== Regression ===")
print(f"RMSE: {rmse:.2f} minutes")
print(f"R\u00b2:   {r2:.4f}")

In [ ]:
# Cell 12 — Feature importances
imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS_FULL).sort_values(ascending=False)
print("Feature importances (classifier):")
print(imp.to_string())

In [ ]:
# Cell 13 — Save all artifacts

joblib.dump(clf,           CLASSIFIER_PATH)
joblib.dump(reg,           REGRESSOR_PATH)
joblib.dump(enc,           ENCODER_PATH)
joblib.dump(top_orig_list, TOP_ORIG_PATH)
joblib.dump(top_dest_list, TOP_DEST_PATH)
joblib.dump({'route_avg': route_avg, 'global_avg': global_avg}, ROUTE_AVG_PATH)
joblib.dump({
    'feature_cols': FEATURE_COLS_FULL,
    'cat_cols':     CAT_COLS,
    'num_cols':     NUM_COLS,
    'num_medians':  num_medians
}, PREPROCESSOR_PATH)

df[FEATURE_COLS + ['DELAY_CLASS','DELAY_MINUTES']].head(10000).to_parquet(PROCESSED_OUT, index=False)

print("Saved artifacts:")
for p in [CLASSIFIER_PATH, REGRESSOR_PATH, ENCODER_PATH,
          TOP_ORIG_PATH, TOP_DEST_PATH, ROUTE_AVG_PATH, PREPROCESSOR_PATH]:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")

print(f"\nFinal RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB")
print("All done!")